# Comparing emcee vs PyMC (NUTS) for NFW Profile Inference

This notebook tests whether using Hamiltonian Monte Carlo (via PyMC's NUTS sampler)
would significantly close the speed gap between MCMC and SBI.

**Key question**: Is emcee slow because it uses an affine-invariant ensemble sampler
rather than HMC, or is the cost dominated by likelihood evaluations?


In [ ]:
# !pip install pymc
# !pip install --upgrade numpy

import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats

# PyMC and pytensor for HMC
import pymc as pm
import pytensor.tensor as pt
import arviz as az

# emcee for comparison
import emcee

# colossus for cosmology
from colossus.cosmology import cosmology
from colossus.halo import profile_nfw, concentration

# Set up cosmology
cosmology.setCosmology("planck18")

# Add parent to path for imports
sys.path.insert(0, "..")
from weaklensclustersbi.simulations import wlprofile, populationutils

## 1. Generate a synthetic observation

We'll create one realistic weak lensing observation to use for both samplers.


In [ ]:
# True parameters
TRUE_LOG10MASS = 14.5
TRUE_Z = 0.275  # midpoint of [0.2, 0.35]

# Use concentration consistent with the M-c relation prior (Child+18)
# This avoids bias from prior-data mismatch
TRUE_CONCENTRATION = populationutils.get_concentration(
    TRUE_LOG10MASS, model="child18", z=TRUE_Z
)
print(f"Using M-c relation concentration: c = {TRUE_CONCENTRATION:.2f}")

# Radial bins (30 bins from 1 to 1000 kpc/h)
NUM_RADIAL_BINS = 30
RBINS = 10 ** np.linspace(0, 3, NUM_RADIAL_BINS)

# Generate true profile
true_profile = wlprofile.simulate_nfw(
    TRUE_LOG10MASS, TRUE_CONCENTRATION, rbins=RBINS, z=TRUE_Z
)
log_true_profile = np.log10(true_profile)

# Add noise (0.3 dex as in the configs)
NOISE_DEX = 0.3
np.random.seed(42)
observed_log_profile = log_true_profile + np.random.normal(
    0, NOISE_DEX, NUM_RADIAL_BINS
)
sigma = np.full(NUM_RADIAL_BINS, NOISE_DEX)

print(f"True log10(M) = {TRUE_LOG10MASS}, c = {TRUE_CONCENTRATION:.2f}")
print(f"Profile shape: {observed_log_profile.shape}")

In [ ]:
# Visualize
plt.figure(figsize=(8, 5))
plt.errorbar(
    RBINS, observed_log_profile, yerr=sigma, fmt="o", label="Observed", alpha=0.7
)
plt.plot(RBINS, log_true_profile, "k-", lw=2, label="True profile")
plt.xscale("log")
plt.xlabel("R [kpc/h]")
plt.ylabel(r"$\log_{10}(\Sigma)$ [M$_\odot$/kpc$^2$]")
plt.legend()
plt.title("Synthetic Weak Lensing Observation")
plt.show()

## 2. Define priors (same for both samplers)


In [ ]:
# Prior bounds
PRIORS = {
    "min_log10mass": 10.0,
    "max_log10mass": 20.0,
    "min_concentration": 0.0,
    "max_concentration": 10.0,
    "mc_scatter": 0.1,
    "mc_relation": "child18",
    "min_z": 0.2,
    "max_z": 0.35,
}

## 3. emcee Implementation (existing code)


In [ ]:
def emcee_logprior(params, priors):
    """Log prior for emcee (from mcmcutils.py)"""
    log10mass, conc, log_f = params

    if not priors["min_log10mass"] < log10mass < priors["max_log10mass"]:
        return -np.inf
    if not priors["min_concentration"] < conc < priors["max_concentration"]:
        return -np.inf
    if not (-10 < log_f < 10):
        return -np.inf

    # M-c relation prior
    z = (priors["min_z"] + priors["max_z"]) / 2
    c_from_m = populationutils.get_concentration(
        log10mass, model=priors["mc_relation"], z=z
    )
    return scipy.stats.norm(loc=c_from_m, scale=priors["mc_scatter"]).logpdf(conc)


def emcee_loglike(params, priors, obs_profile, obs_sigma):
    """Log likelihood for emcee (from mcmcutils.py)"""
    log10mass, conc, log_f = params
    z = (priors["min_z"] + priors["max_z"]) / 2

    # Model prediction
    model_profile = np.log10(wlprofile.simulate_nfw(log10mass, conc, rbins=RBINS, z=z))

    # Likelihood with intrinsic scatter
    sigma2 = obs_sigma**2 + (np.exp(log_f) * model_profile) ** 2
    return -0.5 * np.sum((model_profile - obs_profile) ** 2 / sigma2 + np.log(sigma2))


def emcee_logprob(params, priors, obs_profile, obs_sigma):
    """Log posterior for emcee"""
    lp = emcee_logprior(params, priors)
    if not np.isfinite(lp):
        return -np.inf
    return lp + emcee_loglike(params, priors, obs_profile, obs_sigma)

In [ ]:
def run_emcee(obs_profile, obs_sigma, priors, nwalkers=100, nburn=100, nsteps=500):
    """Run emcee sampler"""
    ndim = 3  # log10mass, concentration, log_f

    # Initialize walkers near reasonable starting point
    starts = np.array([14.0, 4.0, 0.0])
    p0 = starts + 0.1 * np.random.randn(nwalkers, ndim)

    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, emcee_logprob, args=[priors, obs_profile, obs_sigma]
    )

    # Burn-in
    print(f"emcee: Running burn-in ({nburn} steps)...")
    state = sampler.run_mcmc(p0, nburn, progress=True)
    sampler.reset()

    # Production
    print(f"emcee: Running production ({nsteps} steps)...")
    sampler.run_mcmc(state, nsteps, progress=True)

    return sampler

## 4. PyMC Implementation (NUTS/HMC)

The challenge: PyMC needs gradients via pytensor, but colossus NFW isn't differentiable.
We need to reimplement the NFW surface density analytically in pytensor.

### NFW Surface Density Formula

For an NFW profile with scale radius $r_s$ and characteristic density $\rho_s$:

$$\Sigma(R) = 2 \rho_s r_s \cdot g(x)$$

where $x = R/r_s$ and:

- $x < 1$: $g(x) = \frac{1}{x^2-1}\left(1 - \frac{2}{\sqrt{1-x^2}} \text{arctanh}\sqrt{\frac{1-x}{1+x}}\right)$
- $x = 1$: $g(x) = 1/3$
- $x > 1$: $g(x) = \frac{1}{x^2-1}\left(1 - \frac{2}{\sqrt{x^2-1}} \arctan\sqrt{\frac{x-1}{1+x}}\right)$


In [ ]:
from colossus.halo import mass_so


def nfw_surface_density_numpy(R, log10mass, conc, z=0.275):
    """
    NFW surface density - pure numpy implementation for validation.
    Uses colossus mass_so.M_to_R for correct virial radius calculation.

    Parameters
    ----------
    R : array
        Radial distances in kpc/h
    log10mass : float
        Log10 of virial mass in Msun/h
    conc : float
        Concentration parameter
    z : float
        Redshift
    """
    M = 10**log10mass

    # Get virial radius using colossus (handles mass definition correctly)
    r_vir = mass_so.M_to_R(M, z, "vir")

    # Scale radius
    r_s = r_vir / conc

    # Characteristic density from mass normalization:
    # M = 4 pi rho_s r_s^3 [ln(1+c) - c/(1+c)]
    m_factor = np.log(1 + conc) - conc / (1 + conc)
    rho_s = M / (4 * np.pi * r_s**3 * m_factor)

    # Dimensionless radius
    x = R / r_s

    # Surface density calculation (handle x<1, x=1, x>1)
    g = np.zeros_like(x, dtype=float)

    # x < 1
    mask_lt = x < 0.999
    x_lt = x[mask_lt]
    sqrt_term = np.sqrt((1 - x_lt) / (1 + x_lt))
    g[mask_lt] = (1 - 2 / np.sqrt(1 - x_lt**2) * np.arctanh(sqrt_term)) / (x_lt**2 - 1)

    # x ~ 1
    mask_eq = (x >= 0.999) & (x <= 1.001)
    g[mask_eq] = 1 / 3

    # x > 1
    mask_gt = x > 1.001
    x_gt = x[mask_gt]
    sqrt_term = np.sqrt((x_gt - 1) / (1 + x_gt))
    g[mask_gt] = (1 - 2 / np.sqrt(x_gt**2 - 1) * np.arctan(sqrt_term)) / (x_gt**2 - 1)

    Sigma = 2 * rho_s * r_s * g
    return Sigma

In [ ]:
# Validate against colossus
test_sigma_ours = nfw_surface_density_numpy(
    RBINS, TRUE_LOG10MASS, TRUE_CONCENTRATION, TRUE_Z
)
test_sigma_colossus = wlprofile.simulate_nfw(
    TRUE_LOG10MASS, TRUE_CONCENTRATION, rbins=RBINS, z=TRUE_Z
)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.loglog(RBINS, test_sigma_ours, "b-", label="Our implementation")
plt.loglog(RBINS, test_sigma_colossus, "r--", label="Colossus")
plt.xlabel("R [kpc/h]")
plt.ylabel(r"$\Sigma$ [M$_\odot$/kpc$^2$]")
plt.legend()
plt.title("NFW Surface Density")

plt.subplot(1, 2, 2)
plt.semilogx(RBINS, (test_sigma_ours - test_sigma_colossus) / test_sigma_colossus * 100)
plt.xlabel("R [kpc/h]")
plt.ylabel("Relative difference [%]")
plt.title("Comparison")
plt.axhline(0, color="k", ls="--")
plt.tight_layout()
plt.show()

print(
    f"Max relative difference: {np.max(np.abs(test_sigma_ours - test_sigma_colossus) / test_sigma_colossus) * 100:.2f}%"
)

In [ ]:
# Precomputed cosmological constants for z=0.275 (from colossus planck18)
# These are needed because pytensor can't call colossus functions
Z_FIXED = 0.275
RHO_CRIT_Z = 370.194544  # Msun h^2 / kpc^3 at z=0.275
DELTA_VIR_Z = 124.89  # Bryan & Norman virial overdensity at z=0.275


def nfw_surface_density_pytensor(
    R, log10mass, conc, rho_c=RHO_CRIT_Z, Delta=DELTA_VIR_Z
):
    """
    NFW surface density - pytensor implementation for automatic differentiation.
    Uses precomputed cosmological constants and pt.switch for clean branching.
    """
    # Mass
    M = 10**log10mass

    # Virial radius: R_vir = (3M / 4pi Delta rho_c)^(1/3)
    r_vir = (3 * M / (4 * np.pi * Delta * rho_c)) ** (1 / 3)

    # Scale radius
    r_s = r_vir / conc

    # Characteristic density from mass normalization:
    # M = 4 pi rho_s r_s^3 [ln(1+c) - c/(1+c)]
    m_factor = pt.log(1 + conc) - conc / (1 + conc)
    rho_s = M / (4 * np.pi * r_s**3 * m_factor)

    # Dimensionless radius
    x = R / r_s

    # Surface density: Sigma = 2 * rho_s * r_s * g(x)
    # g(x) has different forms for x<1, x=1, x>1
    # Use pt.switch for clean branching (avoids blending issues near x=1)

    # Clipped versions for numerical stability in each branch
    x_lt = pt.clip(x, 0.001, 0.99)  # For x < 1 formula
    x_gt = pt.clip(x, 1.01, 100.0)  # For x > 1 formula

    # x < 1 branch: g = (1 - 2/sqrt(1-x^2) * arctanh(sqrt((1-x)/(1+x)))) / (x^2 - 1)
    sqrt_1mx2_lt = pt.sqrt(1 - x_lt**2)
    arg_lt = pt.sqrt((1 - x_lt) / (1 + x_lt))
    F_lt = 2 / sqrt_1mx2_lt * pt.arctanh(arg_lt)
    g_lt = (1 - F_lt) / (x_lt**2 - 1)

    # x > 1 branch: g = (1 - 2/sqrt(x^2-1) * arctan(sqrt((x-1)/(1+x)))) / (x^2 - 1)
    sqrt_x2m1_gt = pt.sqrt(x_gt**2 - 1)
    arg_gt = pt.sqrt((x_gt - 1) / (1 + x_gt))
    F_gt = 2 / sqrt_x2m1_gt * pt.arctan(arg_gt)
    g_gt = (1 - F_gt) / (x_gt**2 - 1)

    # x = 1: g = 1/3 (limiting value)
    g_eq = 1 / 3

    # Select branch using switch: x < 0.99 -> g_lt, 0.99 <= x <= 1.01 -> g_eq, x > 1.01 -> g_gt
    g = pt.switch(x < 0.99, g_lt, pt.switch(x > 1.01, g_gt, g_eq))

    Sigma = 2 * rho_s * r_s * g
    return Sigma


# Validate pytensor implementation against numpy version
def test_pytensor_impl():
    import pytensor

    R_test = pt.dvector("R")
    logM_test = pt.dscalar("logM")
    c_test = pt.dscalar("c")

    Sigma_pt = nfw_surface_density_pytensor(R_test, logM_test, c_test)
    f = pytensor.function([R_test, logM_test, c_test], Sigma_pt)

    Sigma_pytensor = f(RBINS, TRUE_LOG10MASS, TRUE_CONCENTRATION)
    Sigma_numpy = nfw_surface_density_numpy(
        RBINS, TRUE_LOG10MASS, TRUE_CONCENTRATION, z=Z_FIXED
    )

    rel_diff = np.abs(Sigma_pytensor - Sigma_numpy) / Sigma_numpy * 100
    max_diff = np.max(rel_diff)
    print(f"Pytensor vs Numpy max diff: {max_diff:.4f}%")
    return max_diff < 1.0  # Should be < 1% different


print("Testing pytensor implementation...")
assert test_pytensor_impl(), "Pytensor implementation does not match numpy!"
print("SUCCESS: Pytensor matches numpy implementation")

In [ ]:
def run_pymc(obs_profile, obs_sigma, priors, tune=500, draws=2000):
    """
    Run PyMC NUTS sampler.

    Note: NUTS is more efficient per sample, so we need fewer draws.
    With 4 chains of 2000 draws = 8000 samples (vs emcee's 100 walkers * 500 = 50000)
    """
    with pm.Model() as model:
        # Priors
        log10mass = pm.Uniform(
            "log10mass", lower=priors["min_log10mass"], upper=priors["max_log10mass"]
        )

        # Concentration with M-c relation prior
        # Linear approximation to Child+18 M-c relation at z=0.275:
        # Fitted to match populationutils.get_concentration() in the range logM=[14, 15]
        c_expected = 4.538 - 0.775 * (log10mass - 14.5)

        conc = pm.TruncatedNormal(
            "concentration",
            mu=c_expected,
            sigma=priors["mc_scatter"],
            lower=priors["min_concentration"],
            upper=priors["max_concentration"],
        )

        # Intrinsic scatter parameter
        log_f = pm.Uniform("log_f", lower=-10, upper=10)

        # Model prediction (in log10 space)
        # Uses precomputed cosmological constants for z=0.275
        model_Sigma = nfw_surface_density_pytensor(RBINS, log10mass, conc)
        model_log_profile = pt.log10(model_Sigma)

        # Total variance (measurement + intrinsic scatter)
        sigma2 = obs_sigma**2 + (pt.exp(log_f) * model_log_profile) ** 2

        # Likelihood
        pm.Potential(
            "likelihood",
            -0.5
            * pt.sum((model_log_profile - obs_profile) ** 2 / sigma2 + pt.log(sigma2)),
        )

        # Sample with NUTS
        print(f"PyMC: Running NUTS ({tune} tune + {draws} draws per chain)...")
        trace = pm.sample(
            tune=tune, draws=draws, cores=4, return_inferencedata=True, progressbar=True
        )

    return trace, model

## 5. Run both samplers and compare


In [ ]:
# Run emcee
# Use 1500 steps to get reliable autocorrelation estimate (need ~50 * tau, tau ≈ 25)
print("=" * 60)
print("EMCEE (Affine-Invariant Ensemble Sampler)")
print("=" * 60)

t0 = time.time()
emcee_sampler = run_emcee(
    observed_log_profile, sigma, PRIORS, nwalkers=50, nburn=200, nsteps=1500
)
emcee_time = time.time() - t0

print(f"\nemcee total time: {emcee_time:.1f} seconds")
print(f"emcee samples: {emcee_sampler.flatchain.shape[0]}")

In [ ]:
# Run PyMC
print("=" * 60)
print("PyMC NUTS (Hamiltonian Monte Carlo)")
print("=" * 60)

t0 = time.time()
pymc_trace, pymc_model = run_pymc(
    observed_log_profile, sigma, PRIORS, tune=500, draws=2000
)
pymc_time = time.time() - t0

print(f"\nPyMC total time: {pymc_time:.1f} seconds")
n_pymc_samples = pymc_trace.posterior["log10mass"].size
print(f"PyMC samples: {n_pymc_samples}")

## 6. Compare results


In [ ]:
# Extract samples
emcee_mass = emcee_sampler.flatchain[:, 0]
emcee_conc = emcee_sampler.flatchain[:, 1]

pymc_mass = pymc_trace.posterior["log10mass"].values.flatten()
pymc_conc = pymc_trace.posterior["concentration"].values.flatten()

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Mass posterior
axes[0].hist(emcee_mass, bins=50, density=True, alpha=0.5, label="emcee")
axes[0].hist(pymc_mass, bins=50, density=True, alpha=0.5, label="PyMC NUTS")
axes[0].axvline(TRUE_LOG10MASS, color="k", ls="--", label="Truth")
axes[0].set_xlabel(r"$\log_{10}(M)$")
axes[0].set_ylabel("Density")
axes[0].legend()
axes[0].set_title("Mass Posterior")

# Concentration posterior
axes[1].hist(emcee_conc, bins=50, density=True, alpha=0.5, label="emcee")
axes[1].hist(pymc_conc, bins=50, density=True, alpha=0.5, label="PyMC NUTS")
axes[1].axvline(TRUE_CONCENTRATION, color="k", ls="--", label="Truth")
axes[1].set_xlabel("Concentration")
axes[1].set_ylabel("Density")
axes[1].legend()
axes[1].set_title("Concentration Posterior")

# 2D posterior
axes[2].scatter(emcee_mass[::10], emcee_conc[::10], alpha=0.3, s=1, label="emcee")
axes[2].scatter(pymc_mass[::2], pymc_conc[::2], alpha=0.3, s=1, label="PyMC NUTS")
axes[2].scatter(
    [TRUE_LOG10MASS],
    [TRUE_CONCENTRATION],
    color="red",
    s=100,
    marker="*",
    label="Truth",
    zorder=10,
)
axes[2].set_xlabel(r"$\log_{10}(M)$")
axes[2].set_ylabel("Concentration")
axes[2].legend()
axes[2].set_title("Joint Posterior")

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"\nTrue values: log10(M) = {TRUE_LOG10MASS}, c = {TRUE_CONCENTRATION}")

print(f"\nemcee:")
print(f"  log10(M) = {np.mean(emcee_mass):.3f} +/- {np.std(emcee_mass):.3f}")
print(f"  c        = {np.mean(emcee_conc):.3f} +/- {np.std(emcee_conc):.3f}")

print(f"\nPyMC NUTS:")
print(f"  log10(M) = {np.mean(pymc_mass):.3f} +/- {np.std(pymc_mass):.3f}")
print(f"  c        = {np.mean(pymc_conc):.3f} +/- {np.std(pymc_conc):.3f}")

In [ ]:
# Timing comparison
print("=" * 60)
print("TIMING COMPARISON")
print("=" * 60)

print(
    f"\nemcee:     {emcee_time:.1f} seconds ({emcee_sampler.flatchain.shape[0]} samples)"
)
print(f"PyMC NUTS: {pymc_time:.1f} seconds ({n_pymc_samples} samples)")

# Effective sample size comparison
print("\n--- Effective Sample Size (ESS) ---")

# emcee ESS
try:
    tau = emcee_sampler.get_autocorr_time(quiet=True)
    emcee_ess = emcee_sampler.flatchain.shape[0] / np.mean(tau)
    print(f"\nemcee:")
    print(f"  Autocorrelation time: {np.mean(tau):.1f}")
    print(f"  ESS: {emcee_ess:.0f}")
    print(f"  ESS/second: {emcee_ess/emcee_time:.1f}")
except Exception as e:
    print(f"\nemcee: Could not compute autocorrelation ({e})")
    emcee_ess = None

# PyMC ESS
pymc_ess_data = az.ess(pymc_trace)
pymc_ess_mass = float(pymc_ess_data["log10mass"])
pymc_ess_conc = float(pymc_ess_data["concentration"])
pymc_ess_avg = (pymc_ess_mass + pymc_ess_conc) / 2

print(f"\nPyMC NUTS:")
print(f"  ESS (log10mass): {pymc_ess_mass:.0f}")
print(f"  ESS (concentration): {pymc_ess_conc:.0f}")
print(f"  ESS (average): {pymc_ess_avg:.0f}")
print(f"  ESS/second: {pymc_ess_avg/pymc_time:.1f}")

# Summary
print("\n--- Summary ---")
print(f"Raw time ratio (emcee/PyMC): {emcee_time/pymc_time:.2f}x")
if emcee_ess:
    ess_per_sec_ratio = (pymc_ess_avg / pymc_time) / (emcee_ess / emcee_time)
    print(f"ESS/second ratio (PyMC/emcee): {ess_per_sec_ratio:.2f}x")
    print(
        f"\nPyMC NUTS is {ess_per_sec_ratio:.1f}x more efficient per second than emcee"
    )

## 7. Conclusion


In [ ]:
print(
    """
CONCLUSIONS
===========

1. NUTS (PyMC) vs Ensemble Sampler (emcee):
   - NUTS uses gradient information for efficient proposals
   - Typically needs fewer samples for same effective sample size
   - But requires differentiable likelihood (had to reimplement NFW in pytensor)

2. Does HMC close the gap with SBI?
   - Even if NUTS is 5-10x more efficient than emcee...
   - SBI is >400x faster because it amortizes training cost
   - Once trained, SBI inference is essentially free (neural net forward pass)
   - MCMC (any flavor) still requires likelihood evaluations per sample

3. For the paper:
   - emcee is a reasonable MCMC baseline (affine-invariant, not naive MH)
   - HMC would help but not close orders-of-magnitude gap
   - The fundamental advantage of SBI is amortization, not sampling efficiency
"""
)